In [1]:
from dataclasses import dataclass
from pathlib import Path
import os

In [2]:
CONFIG_FILE_PATH = Path("config/config.yaml")
PARAMS_FILE_PATH = Path("params.yaml")
GLOBAL_PATH = Path("/Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer")

In [3]:
@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_url: str
    local_data_file: Path
    unzip_dir: Path

In [4]:
import os
from typing import Any, Dict, Optional
from box.exceptions import BoxValueError
from ContentSummarizer.utils.logger import logger

from ensure import ensure_annotations
from box import ConfigBox
from pathlib import Path
import yaml



@ensure_annotations
def read_yaml(path_to_yaml: Path) -> ConfigBox:
    """
    Reads a YAML file and returns its contents as a ConfigBox object.

    Args:
        path_to_yaml (Path): Path to YAML file

    Returns:
        ConfigBox: Parsed YAML content
    """

    try:

        resolved_path = Path(GLOBAL_PATH, path_to_yaml).resolve()

        logger.info(
            "Reading YAML file: %s",
            resolved_path
        )

        logger.info(
            "Current Working Directory: %s",
            Path.cwd()
        )

        if not resolved_path.exists():

            logger.error(
                "YAML file does not exist: %s",
                resolved_path
            )

            raise FileNotFoundError(
                f"YAML file not found: {resolved_path}"
            )

        with open(
            resolved_path,
            "r",
            encoding="utf-8"
        ) as yaml_file:

            yaml_content = yaml.safe_load(yaml_file)

        logger.info(
            "Successfully loaded YAML file: %s",
            resolved_path
        )

        return ConfigBox(yaml_content)

    except Exception as e:

        logger.exception(
            "Error reading YAML file: %s",
            path_to_yaml
        )

        raise BoxValueError(
            f"Error reading YAML file at {path_to_yaml}: {e}"
        )
    
def create_directories(path_to_directories: list) -> None:

    try:

        logger.info(
            "Current Working Directory: %s",
            Path.cwd()
        )

        for path in path_to_directories:

            resolved_path = Path(GLOBAL_PATH, path).resolve()

            logger.info(
                "Original Path: %s | Resolved Path: %s",
                path,
                resolved_path
            )

            if resolved_path.is_file():

                logger.warning(
                    "Path %s is a file. Skipping.",
                    resolved_path
                )

                continue

            resolved_path.mkdir(
                parents=True,
                exist_ok=True
            )

            logger.info(
                "Directory created at: %s",
                resolved_path
            )

    except Exception as e:

        logger.exception(
            "Error creating directories: %s",
            str(e)
        )

        raise BoxValueError(
            f"Error creating directories: {e}"
        )
    
@ensure_annotations
def get_size(path: Path) -> str:
    """
    Gets the size of a file or directory in a human-readable format.

    Args:
        path (Path): The path to the file or directory.
    Returns:
        str: The size of the file or directory in a human-readable format.
    """
    try:
        if Path(GLOBAL_PATH, path).is_file():
            size = Path(GLOBAL_PATH, path).stat().st_size
        else:
            size = sum(Path(GLOBAL_PATH, f).stat().st_size for f in Path(GLOBAL_PATH, path).glob('**/*') if Path(GLOBAL_PATH, f).is_file())
        
        for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
            if size < 1024:
                return f"{size:.2f} {unit}"
            size /= 1024
        return f"{size:.2f} PB"
    except Exception as e:
        logger.error(f"Error getting size for {path}: {e}")
        raise BoxValueError(f"Error getting size for {path}: {e}")

[2026-06-03 11:13:33] [INFO] [content_summarizer] [logger.py:102] Logger initialized successfully


In [5]:
# Read config.yaml file
# from ContentSummarizer.constants import CONFIG_FILE_PATH, GLOBAL_PATH


config = read_yaml(Path(CONFIG_FILE_PATH))
# Create necessary directories
create_directories([config["artifacts_root"], config["data_ingestion"]["root_dir"]])

[2026-06-03 11:13:34] [INFO] [content_summarizer] [2511429230.py:29] Reading YAML file: /Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer/config/config.yaml
[2026-06-03 11:13:34] [INFO] [content_summarizer] [2511429230.py:34] Current Working Directory: /Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer/research
[2026-06-03 11:13:34] [INFO] [content_summarizer] [2511429230.py:58] Successfully loaded YAML file: /Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer/config/config.yaml
[2026-06-03 11:13:34] [INFO] [content_summarizer] [2511429230.py:80] Current Working Directory: /Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer/research
[2026-06-03 11:13:34] [INFO] [content_summarizer] [2511429230.py:89] Original Path: artifacts | Resolved Path: /Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarize

In [ ]:
class ConfigurationManager:
    config_file_path: Path
    params_file_path: Path

    def __init__(self, config_file_path: Path = Path(GLOBAL_PATH, CONFIG_FILE_PATH), params_file_path: Path = Path(GLOBAL_PATH, PARAMS_FILE_PATH)):
        self.config_file_path = Path(GLOBAL_PATH, config_file_path)
        self.params_file_path = Path(GLOBAL_PATH, params_file_path)
        self.config = read_yaml(self.config_file_path)
        self.params = read_yaml(self.params_file_path)
        create_directories([self.config["artifacts_root"]])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        data_ingestion_config = self.config["data_ingestion"]
        create_directories([data_ingestion_config["root_dir"]])
        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(data_ingestion_config["root_dir"]),
            source_url=data_ingestion_config["source_url"],
            local_data_file=Path(data_ingestion_config["local_data_file"]),
            unzip_dir=Path(data_ingestion_config["unzip_dir"])
        )
        return data_ingestion_config

In [7]:
import os
import zipfile
import requests

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        response = requests.get(self.config.source_url)
        response.raise_for_status()  # Check if the request was successful
        with open(Path(GLOBAL_PATH, self.config.local_data_file), 'wb') as file:
            file.write(response.content)
        print(f"File downloaded successfully and saved to {Path(GLOBAL_PATH, self.config.local_data_file)}")

    def unzip_file(self):
        with zipfile.ZipFile(Path(GLOBAL_PATH, self.config.local_data_file), 'r') as zip_ref:
            zip_ref.extractall(Path(GLOBAL_PATH, self.config.unzip_dir))
        print(f"File unzipped successfully to {Path(GLOBAL_PATH, self.config.unzip_dir)}")

    def initiate_data_ingestion(self):
        self.download_file()
        self.unzip_file()

    def test_connection(self):
        try:
            response = requests.get(self.config.source_url)
            response.raise_for_status()
            print("Connection successful!")
        except requests.exceptions.RequestException as e:
            print(f"Connection failed: {e}")
    
    def test_unzip(self):
        try:
            with zipfile.ZipFile(Path(GLOBAL_PATH, self.config.local_data_file), 'r') as zip_ref:
                zip_ref.extractall(Path(GLOBAL_PATH, self.config.unzip_dir))
            print("Unzip successful!")
        except zipfile.BadZipFile as e:
            print(f"Unzip failed: {e}")

    def check_file_exists(self):
        if os.path.exists(Path(GLOBAL_PATH, self.config.local_data_file)):
            print(f"File exists at {Path(GLOBAL_PATH, self.config.local_data_file)}")
        else:
            print(f"File does not exist at {Path(GLOBAL_PATH, self.config.local_data_file)}")

    def show_sample_ingested_data(self):
        ingested_files = os.listdir(Path(GLOBAL_PATH, self.config.unzip_dir))
        print(f"Ingested files: {ingested_files}")

In [8]:
config = ConfigurationManager(Path(CONFIG_FILE_PATH), Path(PARAMS_FILE_PATH))
data_ingestion_config = config.get_data_ingestion_config()
print(data_ingestion_config)

[2026-06-03 11:13:36] [INFO] [content_summarizer] [2511429230.py:29] Reading YAML file: /Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer/config/config.yaml
[2026-06-03 11:13:36] [INFO] [content_summarizer] [2511429230.py:34] Current Working Directory: /Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer/research
[2026-06-03 11:13:36] [INFO] [content_summarizer] [2511429230.py:58] Successfully loaded YAML file: /Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer/config/config.yaml
[2026-06-03 11:13:36] [INFO] [content_summarizer] [2511429230.py:29] Reading YAML file: /Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer/params.yaml
[2026-06-03 11:13:36] [INFO] [content_summarizer] [2511429230.py:34] Current Working Directory: /Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer/research
[2026-06-

In [9]:
dataIngestion = DataIngestion(config.get_data_ingestion_config())
dataIngestion.test_connection()
dataIngestion.initiate_data_ingestion()

[2026-06-03 11:13:36] [INFO] [content_summarizer] [2511429230.py:80] Current Working Directory: /Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer/research
[2026-06-03 11:13:36] [INFO] [content_summarizer] [2511429230.py:89] Original Path: artifacts/data_ingestion | Resolved Path: /Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer/artifacts/data_ingestion
[2026-06-03 11:13:36] [INFO] [content_summarizer] [2511429230.py:109] Directory created at: /Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer/artifacts/data_ingestion


Connection successful!
File downloaded successfully and saved to /Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer/artifacts/data_ingestion/data.zip
File unzipped successfully to /Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer/artifacts/data_ingestion


In [10]:
dataIngestion.check_file_exists()
dataIngestion.test_unzip()
dataIngestion.show_sample_ingested_data()

File exists at /Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer/artifacts/data_ingestion/data.zip


Unzip successful!
Ingested files: ['data.zip', 'cnn_dailymail']
